In [1]:
import os
import pandas as pd
import numpy as np

# Define the input and output directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Imputed_Data_KNN'
output_base_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Input_Data_with_Missing'

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to calculate missing percentage
def calculate_missing_percentage(df, column):
    total_values = len(df)
    missing_values = df[column].isnull().sum()
    missing_percentage = (missing_values / total_values) * 100
    return missing_percentage, missing_values, total_values

# Main function to process the data
def process_data(column_name, missing_percentage, run_number):
    # Create the output directory for this run
    output_directory = os.path.join(output_base_directory, f'Run{run_number}_data')
    os.makedirs(output_directory, exist_ok=True)
    
    # Traverse the directory and process each CSV file
    for filename in os.listdir(input_directory):
        if filename.endswith('.csv'):
            study_site = extract_study_site(filename)
            filepath = os.path.join(input_directory, filename)

            # Load the CSV file into a DataFrame
            df = pd.read_csv(filepath)

            # Convert datetime column to datetime type
            df['datetime'] = pd.to_datetime(df['datetime'])

            # Calculate initial missing percentage of the specified column
            initial_missing_percentage, initial_missing_count, total_values = calculate_missing_percentage(df, column_name)

            # Create a column to mark originally missing data
            df['missing_info'] = np.where(df[column_name].isnull(), 'OM', '')

            # Determine the number of additional values to remove to achieve the specified missing percentage
            target_missing_count = int(missing_percentage * total_values / 100)
            additional_missing_count = target_missing_count - initial_missing_count

            # Create a column to store removed values
            df['removed_values'] = np.nan

            if additional_missing_count > 0:
                # Randomly select indices to set as NaN
                non_missing_indices = df[df[column_name].notnull()].index
                if len(non_missing_indices) < additional_missing_count:
                    print(f"Not enough non-missing values in {filename} to achieve {missing_percentage}% missing for column {column_name}")
                    continue
                additional_missing_indices = np.random.choice(non_missing_indices, additional_missing_count, replace=False)
                df.loc[additional_missing_indices, 'removed_values'] = df.loc[additional_missing_indices, column_name]
                df.loc[additional_missing_indices, column_name] = np.nan

                # Mark the artificially missing data
                df.loc[additional_missing_indices, 'missing_info'] = 'AM'

            # Add columns for hour, day, and month
            df['Hour'] = df['datetime'].dt.hour
            df['Day'] = df['datetime'].dt.day
            df['Month'] = df['datetime'].dt.month

            # Save the modified DataFrame to the output directory with the same filename
            output_filepath = os.path.join(output_directory, filename)
            df.to_csv(output_filepath, index=False)
            print(f"Run {run_number}: Processed and saved file: {output_filepath} with {missing_percentage}% missing values in column {column_name}")

# Run the script 10 times with specified missing percentage
column_name = 'PM2.5'  # You can change this to any column you need
missing_percentage = 10  # You can change this to any percentage you need
for run_number in range(1, 11):
    process_data(column_name, missing_percentage, run_number)

print("Processing complete.")

Run 1: Processed and saved file: /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Input_Data_with_Missing/Run1_data/AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv with 10% missing values in column PM2.5
Run 1: Processed and saved file: /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Input_Data_with_Missing/Run1_data/AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv with 10% missing values in column PM2.5
Run 1: Processed and saved file: /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Input_Data_with_Missing/Run1_data/AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv with 10% missing values in column PM2.5
Run 1: Processed and saved file: /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Input_Data_with_Missing/Run1_data/AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv with 10% missing values in column PM2.5
Run 1: Processed and saved file: /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Input_Data_with_Missing/Run1_data/AQMS_ARMIDALE_20